Problem Statement Online book retailers and publishing houses face a critical challenge in predicting a book's commercial success before its release or during its early market presence. With thousands of new titles published annually, making accurate revenue forecasts and inventory decisions can mean the difference between profitable operations and costly overstock or stockouts. This project addresses this business need by framing the problem as a regression task to predict a book's popularity and market performance, using "reviews count" as a proxy for commercial success and reader engagement from best selling online books data.

Success will be measured by RMSE (Root Mean Square Error) as the primary metric, providing interpretable error margins in terms of review counts, with R² serving as a secondary metric to explain the variance captured by our model. From a business perspective, a successful model would enable publishers to identify potential bestsellers earlier in their lifecycle, optimize initial print runs, and make data-driven decisions about marketing investments. The business KPI would be a 15% improvement in inventory turnover rate, a 10% reduction in unsold inventory and 10% improvement in marketing ROI, achieved through more accurate sales forecasting based on predicted review volumes.

In [1]:
# ========== CORRECTED DATA DICTIONARY WITH PROPER RATING VALUES ==========
print("\n" + "="*60)
print("DATA DICTIONARY - Best Selling Books 2023 (CORRECTED)")
print("="*60)

import pandas as pd

# Define all columns from the dataset with separate Allowed Values and Range
columns_info = [
    {
        'Variable Name': 'id',
        'Type': 'Categorical',
        'Data Type': 'object/string',
        'Description': 'Unique identifier and rank of the book in the bestseller list',
        'Allowed Values': '#1 to #100 (100 unique values)',
        'Range': '1-100 (after cleaning)',
        'Missing Values': '0 (0.0%)',
        'Sample Values': '#1, #2, #3',
        'Notes': 'Prefix "#" removed during cleaning; converted to integer for analysis'
    },
    {
        'Variable Name': 'book name',
        'Type': 'Categorical',
        'Data Type': 'object/string',
        'Description': 'Title of the book as it appears on Amazon',
        'Allowed Values': '100 unique book titles',
        'Range': 'N/A',
        'Missing Values': '0 (0.0%)',
        'Sample Values': 'Atomic Habits: An Easy & Proven Way to Build Good Habits & Break Bad Ones',
        'Notes': 'Contains special characters and punctuation; some titles include series information'
    },
    {
        'Variable Name': 'author',
        'Type': 'Categorical',
        'Data Type': 'object/string',
        'Description': 'Author(s) of the book',
        'Allowed Values': '84 unique authors',
        'Range': 'N/A',
        'Missing Values': '0 (0.0%)',
        'Sample Values': 'James Clear, Rebecca Yarros, Prince Harry The Duke of Sussex',
        'Notes': 'Some authors appear multiple times (Sarah J. Maas: 5 books, Colleen Hoover: 5 books)'
    },
    {
        'Variable Name': 'rating',
        'Type': 'Numeric',
        'Data Type': 'float64',
        'Description': 'Average customer rating on a 5-star scale',
        'Allowed Values': '0.0 - 5.0 stars (in 0.1 increments)',  # ✅ CORRECTED - theoretical possible values
        'Range': '4.1 to 4.9 stars',  # Actual observed values in dataset
        'Missing Values': '0 (0.0%)',
        'Sample Values': '4.8, 4.7, 4.5',
        'Notes': 'Extracted numeric value from "X out of 5 stars" format; negatively correlated with review count (-0.233); all books in dataset are bestsellers, hence ratings are high'
    },
    {
        'Variable Name': 'reviews count',
        'Type': 'Numeric',
        'Data Type': 'int64',
        'Description': 'Total number of customer reviews received on Amazon',
        'Allowed Values': 'Any positive integer',
        'Range': '3,296 to 653,111 reviews',
        'Missing Values': '0 (0.0%)',
        'Sample Values': '145,747, 395,512, 116,101',
        'Notes': 'Target variable for regression; highly skewed (skewness=1.96); log transformation recommended'
    },
    {
        'Variable Name': 'form',
        'Type': 'Categorical',
        'Data Type': 'object/string',
        'Description': 'Physical format/edition of the book',
        'Allowed Values': 'Hardcover, Paperback, Board book, Cards',
        'Range': 'N/A',
        'Missing Values': '0 (0.0%)',
        'Sample Values': 'Hardcover, Hardcover, Hardcover',
        'Notes': 'Distribution: Hardcover (47%), Paperback (41%), Board book (11%), Cards (1%); paperback format correlates positively with reviews'
    },
    {
        'Variable Name': 'price',
        'Type': 'Numeric',
        'Data Type': 'float64',
        'Description': 'Current selling price in US dollars',
        'Allowed Values': 'Any positive decimal',
        'Range': '$1.88 to $52.62',
        'Missing Values': '0 (0.0%)',
        'Sample Values': '$18.88, $11.05, $11.99',
        'Notes': 'Cleaned from "$" prefix; optimal price point around $10-15 for maximum reviews; weak negative correlation with review count (-0.084)'
    },
    {
        'Variable Name': 'reading age',
        'Type': 'Categorical',
        'Data Type': 'object/string',
        'Description': 'Recommended age range for readers (if specified)',
        'Allowed Values': 'baby and up, 1-2 years, 3-4 years, 3 years and up, 4 years and up, 5 years and up, 7 years and up, 8 years and up, 9 years and up, 12 years and up, 18 years and up',
        'Range': 'N/A',
        'Missing Values': '78 (78.0%)',
        'Sample Values': 'NaN, NaN, NaN',
        'Notes': 'High missing rate; primarily specified for children\'s books; not used in final modeling'
    },
    {
        'Variable Name': 'print length',
        'Type': 'Numeric',
        'Data Type': 'int64',
        'Description': 'Number of pages in the book',
        'Allowed Values': 'Any positive integer',
        'Range': '24 to 2,896 pages',
        'Missing Values': '57 (57.0%)',
        'Sample Values': '320, 640, 416',
        'Notes': 'Missing for 57 books (box sets, cards); weak positive correlation with review count (0.144)'
    },
    {
        'Variable Name': 'publishing date',
        'Type': 'Datetime',
        'Data Type': 'datetime64',
        'Description': 'Original publication date of the book',
        'Allowed Values': 'Valid calendar dates',
        'Range': '1990 to 2024',
        'Missing Values': '0 (0.0%) after cleaning',
        'Sample Values': '2018-10-16, 2023-11-07, 2023-01-10',
        'Notes': 'Mixed formats (DD/MM/YYYY and Month DD, YYYY); cleaned to datetime; used to extract year, month, days since published'
    },
    {
        'Variable Name': 'genre',
        'Type': 'Categorical',
        'Data Type': 'object/string',
        'Description': 'Primary book category/genre classification',
        'Allowed Values': '16 unique genres (see Notes for list)',
        'Range': 'N/A',
        'Missing Values': '0 (0.0%)',
        'Sample Values': 'Self-Improvement, Fiction & Action & Adventure, Biographies & Memoirs',
        'Notes': 'Genres: Fiction & Action & Adventure (33), Reading & Writing (18), Self-Improvement (8), Biographies & Memoirs (7), Mystery & Thriller & Suspense (6), Health & Fitness & Dieting (5), Business & Money (5), Politics & Social Sciences (4), Cookbooks & Food (3), Arts & Music & Photography (3), Comics & Graphic Novels (2), Puzzles & Games (1), Humor & Entertainment (1), Schools & Teaching (1), Romance (1), Computers & Technology (1)'
    }
]

# Create DataFrame for the data dictionary
data_dict_df = pd.DataFrame(columns_info)

# ========== DISPLAY DATA DICTIONARY ==========
print("\n" + "="*100)
print("DATA DICTIONARY - Best Selling Books 2023 (CORRECTED)")
print("="*100)
print("\n")

# Set display options for better readability
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.width', 200)

# Display the table
print(data_dict_df.to_string(index=False))

# ========== HIGHLIGHT THE CORRECTION ==========
print("\n" + "="*60)
print("✅ CORRECTION NOTE")
print("="*60)
print("\nRating variable corrected:")
print("  • Allowed Values: 0.0 - 5.0 stars (theoretical possible range)")
print("  • Range: 4.1 to 4.9 stars (actual observed values in this dataset)")
print("\nWhy the difference?")
print("  • This dataset contains ONLY bestselling books")
print("  • Bestsellers naturally have higher ratings (4.1-4.9)")
print("  • The theoretical range (0.0-5.0) includes ratings for ALL books, not just bestsellers")

# Save to CSV
data_dict_df.to_csv('data_dictionary_books.csv', index=False)
print("\n✓ Data dictionary saved to 'data_dictionary_books.csv'")


DATA DICTIONARY - Best Selling Books 2023 (CORRECTED)

DATA DICTIONARY - Best Selling Books 2023 (CORRECTED)


  Variable Name        Type     Data Type                                                   Description                                                                                                                                                      Allowed Values                    Range          Missing Values                                                             Sample Values                                                                                                                                                                                                                                                                                                                                                                                                                                           Notes
             id Categorical object/string Unique identifier and r